In [1]:
import os
import nltk
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize

nltk.download('punkt')

# Load model once (faster)
model = SentenceTransformer("all-MiniLM-L6-v2")

TRANSCRIPT_DIR = "datasets/transcripts"
BLOCK_SIZE = 5

def compute_similarity(transcript, block_size=5):
    sentences = sent_tokenize(transcript)

    # group sentences into blocks
    blocks = [
        " ".join(sentences[i:i+block_size])
        for i in range(0, len(sentences), block_size)
    ]

    # embeddings
    embeddings = model.encode(blocks)

    # cosine similarity between adjacent blocks
    scores = []
    for i in range(len(embeddings) - 1):
        sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
        scores.append(float(sim))

    return blocks, scores


results = {}

# process all transcript files
for file in os.listdir(TRANSCRIPT_DIR):
    if file.endswith(".txt"):
        path = os.path.join(TRANSCRIPT_DIR, file)

        with open(path, "r", encoding="utf-8") as f:
            transcript = f.read()

        blocks, scores = compute_similarity(transcript, BLOCK_SIZE)

        results[file] = {
            "num_blocks": len(blocks),
            "similarity_scores": scores
        }

# print results
for file, data in results.items():
    print("\n======================")
    print("File:", file)
    print("Blocks:", data["num_blocks"])
    print("Similarity Scores:", data["similarity_scores"])

C:\Users\ASHOKA MS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [2]:
import json

with open("generated/similarity_results.json", "w") as f:
    json.dump(results, f, indent=4)

In [2]:
import os
import json
import nltk
import matplotlib.pyplot as plt

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize

nltk.download('punkt')


TRANSCRIPT_DIR = "datasets/transcripts"
OUTPUT_DIR = "output"
BLOCK_SIZE = 5
BOUNDARY_THRESHOLD = 0.5   # topic shift threshold

os.makedirs(OUTPUT_DIR, exist_ok=True)

# load model once
model = SentenceTransformer("all-MiniLM-L6-v2")


def process_transcript(text, block_size=5):
    sentences = sent_tokenize(text)

    blocks = [
        " ".join(sentences[i:i+block_size])
        for i in range(0, len(sentences), block_size)
    ]

    embeddings = model.encode(blocks)

    similarities = []
    for i in range(len(embeddings) - 1):
        sim = cosine_similarity(
            [embeddings[i]],
            [embeddings[i+1]]
        )[0][0]
        similarities.append(float(sim))

    return blocks, similarities


def detect_boundaries(blocks, similarities, threshold):
    boundaries = []
    segments = []

    current_segment = blocks[0]

    for i, score in enumerate(similarities):
        if score < threshold:
            boundaries.append(i + 1)
            segments.append(current_segment)
            current_segment = blocks[i + 1]
        else:
            current_segment += " " + blocks[i + 1]

    segments.append(current_segment)

    return boundaries, segments


def plot_similarity(similarities, file_name):
    plt.figure()
    plt.plot(similarities, marker='o')
    plt.title(f"Similarity Graph: {file_name}")
    plt.xlabel("Block Index")
    plt.ylabel("Cosine Similarity")
    plt.grid(True)

    plt.savefig(os.path.join(OUTPUT_DIR, file_name + "_graph.png"))
    plt.close()



for file in os.listdir(TRANSCRIPT_DIR):
    if not file.endswith(".txt"):
        continue

    path = os.path.join(TRANSCRIPT_DIR, file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    blocks, similarities = process_transcript(text, BLOCK_SIZE)

    boundaries, segments = detect_boundaries(
        blocks, similarities, BOUNDARY_THRESHOLD
    )

    # save raw similarity values
    json_path = os.path.join(OUTPUT_DIR, file + "_similarity.json")
    with open(json_path, "w") as f:
        json.dump(similarities, f, indent=4)

    # save segments
    seg_path = os.path.join(OUTPUT_DIR, file + "_segments.txt")
    with open(seg_path, "w", encoding="utf-8") as f:
        for i, seg in enumerate(segments):
            f.write(f"\n--- Segment {i+1} ---\n")
            f.write(seg + "\n")

    # plot similarity graph
    plot_similarity(similarities, file)

    print("\nProcessed:", file)
    print("Similarity Scores:", similarities)
    print("Topic Boundaries at blocks:", boundaries)
    print("Segments created:", len(segments))

[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [3]:
import os
import json
import re
import numpy as np
import nltk

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize

nltk.download('punkt')

TRANSCRIPT_DIR = "datasets/transcripts"
OUTPUT_FILE = "segmented_output.json"
BLOCK_SIZE = 5

model = SentenceTransformer("all-MiniLM-L6-v2")


# Timestamp Parsing 
def parse_transcript(text):
    pattern = r"\[(\d{2}:\d{2})\]\s*(.*)"
    sentences = []
    times = []

    for line in text.splitlines():
        match = re.match(pattern, line)
        if match:
            times.append(match.group(1))
            sentences.append(match.group(2))
        else:
            sentences.append(line)
            times.append(None)

    # fallback if no timestamps
    if all(t is None for t in times):
        sentences = sent_tokenize(text)
        times = [None] * len(sentences)

    return sentences, times


# Create Blocks 
def create_blocks(sentences, times, block_size):
    blocks = []
    block_times = []

    for i in range(0, len(sentences), block_size):
        block = " ".join(sentences[i:i+block_size])
        blocks.append(block)

        start_time = times[i]
        end_time = times[min(i+block_size-1, len(times)-1)]
        block_times.append((start_time, end_time))

    return blocks, block_times


# Compute Similarity 
def compute_similarities(blocks):
    if len(blocks) < 2:
        return []

    embeddings = model.encode(blocks)

    sims = [
        float(cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0])
        for i in range(len(embeddings)-1)
    ]

    return sims


# Boundary Detection
def detect_boundaries(similarities):
    if len(similarities) == 0:
        return [], 0, 0, 0

    mean = np.mean(similarities)
    std = np.std(similarities)
    threshold = mean - std

    boundaries = [
        i+1 for i, s in enumerate(similarities)
        if s < threshold
    ]

    return boundaries, float(mean), float(std), float(threshold)


#Create Segments
def create_segments(blocks, block_times, boundaries):
    segments = []
    start = 0

    for boundary in boundaries:
        segment_text = " ".join(blocks[start:boundary])
        start_time = block_times[start][0]
        end_time = block_times[boundary-1][1]

        segments.append({
            "start_block": start,
            "end_block": boundary-1,
            "start_time": start_time,
            "end_time": end_time,
            "text": segment_text
        })
        start = boundary

    # last segment
    if len(blocks) > 0:
        segments.append({
            "start_block": start,
            "end_block": len(blocks)-1,
            "start_time": block_times[start][0],
            "end_time": block_times[-1][1],
            "text": " ".join(blocks[start:])
        })

    return segments


# Process Episodes 
all_results = []


files = sorted([f for f in os.listdir(TRANSCRIPT_DIR) if f.endswith(".txt")])

print(f"Found {len(files)} transcript files\n")

for file in files:
    path = os.path.join(TRANSCRIPT_DIR, file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read().strip()

    if len(text) == 0:
        print("Skipping empty file:", file)
        continue

    sentences, times = parse_transcript(text)

    blocks, block_times = create_blocks(sentences, times, BLOCK_SIZE)

    similarities = compute_similarities(blocks)

    boundaries, mean, std, threshold = detect_boundaries(similarities)

    segments = create_segments(blocks, block_times, boundaries)

    print("Processed:", file)
    print("Blocks:", len(blocks), "| Segments:", len(segments))
    print("-" * 40)

    all_results.append({
        "episode": file,
        "num_blocks": len(blocks),
        "similarity_scores": similarities,
        "mean_similarity": mean,
        "std_similarity": std,
        "boundary_threshold": threshold,
        "boundaries": boundaries,
        "segments": segments
    })


# save JSON
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_results, f, indent=4)

print("\n Segmentation complete →", OUTPUT_FILE)

[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [4]:
import os
import json
import re
import numpy as np
import nltk

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize

nltk.download('punkt')


TRANSCRIPT_DIR = "datasets/transcripts"
OUTPUT_FILE = "final_segments.json"

BLOCK_SIZE = 5
K = 1.0   # sensitivity (lower = more segments)

# load SBERT model once
model = SentenceTransformer("all-MiniLM-L6-v2")


# TRANSCRIPT PARSER
def parse_transcript(text):
    """
    Handles:
    ✔ timestamps [12.5] or [00:01:12]
    ✔ plain text
    ✔ whisper transcripts
    ✔ missing punctuation
    """

    text = re.sub(r'\s+', ' ', text).strip()

    # extract timestamps if present
    timestamp_pattern = r"\[(.*?)\]"
    timestamps = re.findall(timestamp_pattern, text)

    # remove timestamps
    clean_text = re.sub(r"\[.*?\]", "", text)

    # sentence tokenize
    sentences = sent_tokenize(clean_text)

    # fallback if tokenizer fails
    if len(sentences) <= 1:
        sentences = clean_text.split(". ")
        sentences = [s.strip() for s in sentences if s.strip()]

    # convert timestamps → seconds
    times = []
    for t in timestamps:
        if ":" in t:
            parts = [float(p) for p in t.split(":")]
            sec = 0
            for p in parts:
                sec = sec * 60 + p
        else:
            try:
                sec = float(t)
            except:
                sec = None
        times.append(sec)

    # if timestamps missing → generate pseudo times
    if len(times) != len(sentences):
        times = list(range(len(sentences)))

    return sentences, times


#  CREATE BLOCKS 
def create_blocks(sentences, times):
    blocks = []
    block_times = []

    for i in range(0, len(sentences), BLOCK_SIZE):
        block_text = " ".join(sentences[i:i+BLOCK_SIZE])
        blocks.append(block_text)

        start_time = times[i]
        end_time = times[min(i+BLOCK_SIZE-1, len(times)-1)]

        block_times.append((start_time, end_time))

    return blocks, block_times


# similarity
def compute_similarity(blocks):
    if len(blocks) < 2:
        return []

    embeddings = model.encode(blocks, convert_to_numpy=True)

    similarities = [
        float(cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0])
        for i in range(len(embeddings)-1)
    ]

    return similarities


# boundary detection
def detect_boundaries(similarities):
    if len(similarities) == 0:
        return [], 0, 0, 0

    mean = np.mean(similarities)
    std = np.std(similarities)

    threshold = mean - (K * std)

    boundaries = [
        i+1 for i, s in enumerate(similarities)
        if s < threshold
    ]

    return boundaries, float(mean), float(std), float(threshold)


# segament creation
def create_segments(blocks, block_times, boundaries):
    segments = []
    start_idx = 0

    for b in boundaries:
        segments.append({
            "start": block_times[start_idx][0],
            "end": block_times[b-1][1],
            "text": " ".join(blocks[start_idx:b])
        })
        start_idx = b

    # last segment
    segments.append({
        "start": block_times[start_idx][0],
        "end": block_times[-1][1],
        "text": " ".join(blocks[start_idx:])
    })

    return segments


# process of all le
all_outputs = []

files = [f for f in os.listdir(TRANSCRIPT_DIR) if f.endswith(".txt")]
print(f"\nFound {len(files)} transcripts\n")

for file in sorted(files):

    path = os.path.join(TRANSCRIPT_DIR, file)

    with open(path, "r", encoding="utf-8") as f:
        text = f.read()

    sentences, times = parse_transcript(text)

    print("Processing:", file)
    print("Sentences:", len(sentences))

    # handle very short transcripts
    if len(sentences) < BLOCK_SIZE:
        print("⚠ Short transcript → single segment created\n")

        segments = [{
            "start": 0,
            "end": len(sentences),
            "text": " ".join(sentences)
        }]

        all_outputs.append({
            "episode": file,
            "segments": segments
        })
        continue

    blocks, block_times = create_blocks(sentences, times)
    similarities = compute_similarity(blocks)

    boundaries, mean, std, threshold = detect_boundaries(similarities)
    segments = create_segments(blocks, block_times, boundaries)

    print("Blocks:", len(blocks))
    print("Segments created:", len(segments))
    print("-" * 40)

    all_outputs.append({
        "episode": file,
        "mean_similarity": mean,
        "std_similarity": std,
        "threshold": threshold,
        "segments": segments
    })



with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(all_outputs, f, indent=4)

print("\n Segmentation complete")
print("Saved to:", OUTPUT_FILE)

[nltk_data] Error loading punkt: <urlopen error [Errno 11001]
[nltk_data]     getaddrinfo failed>
'[Errno 11001] getaddrinfo failed' thrown while requesting HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/adapter_config.json
Retrying in 1s [Retry 1/5].


RuntimeError: Cannot send a request, as the client has been closed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nltk

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import sent_tokenize

nltk.download('punkt')


TRANSCRIPT_FILE = "datasets/transcripts/audio1.txt"
BLOCK_SIZE = 5   # sentences per block

# load SBERT
model = SentenceTransformer("all-MiniLM-L6-v2")


def process_transcript(text):

    sentences = sent_tokenize(text)

    # fallback for transcripts without punctuation
    if len(sentences) <= 1:
        sentences = text.split(". ")
        sentences = [s.strip() for s in sentences if s.strip()]

    blocks = [
        " ".join(sentences[i:i+BLOCK_SIZE])
        for i in range(0, len(sentences), BLOCK_SIZE)
    ]

    embeddings = model.encode(blocks)

    similarities = [
        float(cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0])
        for i in range(len(embeddings)-1)
    ]

    return blocks, similarities


def detect_boundaries(similarities):

    mean = np.mean(similarities)
    std = np.std(similarities)

    threshold = mean - std

    boundaries = [
        i+1 for i, s in enumerate(similarities)
        if s < threshold
    ]

    return boundaries, mean, std, threshold


def create_segments(blocks, boundaries):

    segments = []
    start = 0

    for b in boundaries:
        segments.append(" ".join(blocks[start:b]))
        start = b

    segments.append(" ".join(blocks[start:]))

    return segments



def plot_similarity(similarities, boundaries):

    plt.figure(figsize=(10,4))
    plt.plot(similarities, marker='o')

    # mark detected boundaries
    for b in boundaries:
        plt.axvline(x=b-1, linestyle='--')

    plt.title("Similarity Curve with Detected Topic Boundaries")
    plt.xlabel("Block Index")
    plt.ylabel("Cosine Similarity")
    plt.grid(True)
    plt.show()


# ========= MAIN =========
with open(TRANSCRIPT_FILE, "r", encoding="utf-8") as f:
    transcript = f.read()

blocks, similarities = process_transcript(transcript)

boundaries, mean, std, threshold = detect_boundaries(similarities)

segments = create_segments(blocks, boundaries)

# ===== OUTPUT =====
print("\n========== RESULTS ==========\n")

print("Similarity Score Array:")
print([round(s,3) for s in similarities])

print("\nMean similarity:", round(mean,3))
print("Standard deviation:", round(std,3))
print("Threshold:", round(threshold,3))

print("\nDetected Boundary Indices:")
print(boundaries)

print("\nFinal Segmented Transcript:\n")

for i, seg in enumerate(segments, 1):
    print(f"--- Segment {i} ---")
    print(seg[:400])
    print()

# visualization
plot_similarity(similarities, boundaries)

C:\Users\ASHOKA MS\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package punkt to C:\Users\ASHOKA
[nltk_data]     MS\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 153.92it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
